# Predicting Residential EV Charging Loads using Neural Networks

In [35]:
import numpy as np
import pandas as pd

In [36]:
data = pd.read_csv('Dataset 1_EV charging reports.csv', sep=';')
data.head()


,session_ID,Garage_ID,User_ID,User_type,Shared_ID,Start_plugin,Start_plugin_hour,End_plugout,End_plugout_hour,El_kWh,Duration_hours,month_plugin,weekdays_plugin,Plugin_category,Duration_category
0,1,AdO3,AdO3-4,Private,NaN,21.12.2018 10:20,10,21.12.2018 10:23,10.0,"0,3","0,05",Dec,Friday,late morning (9-12),Less than 3 hours
1,2,AdO3,AdO3-4,Private,NaN,21.12.2018 10:24,10,21.12.2018 10:32,10.0,"0,87","0,136666667",Dec,Friday,late morning (9-12),Less than 3 hours
2,3,AdO3,AdO3-4,Private,NaN,21.12.2018 11:33,11,21.12.2018 19:46,19.0,"29,87","8,216388889",Dec,Friday,late morning (9-12),Between 6 and 9 hours
3,4,AdO3,AdO3-2,Private,NaN,22.12.2018 16:15,16,23.12.2018 16:40,16.0,"15,56","24,41972222",Dec,Saturday,late afternoon (15-18),More than 18 hours
4,5,AdO3,AdO3-2,Private,NaN,24.12.2018 22:03,22,24.12.2018 23:02,23.0,"3,62","0,970555556",Dec,Monday,late evening (21-midnight),Less than 3 hours


In [37]:
print(data.shape)
print(data.dtypes)
print(data.columns.tolist())


(6878, 15)
session_ID             int64
Garage_ID                str
User_ID                  str
User_type                str
Shared_ID                str
Start_plugin             str
Start_plugin_hour      int64
End_plugout              str
End_plugout_hour     float64
El_kWh                   str
Duration_hours           str
month_plugin             str
weekdays_plugin          str
Plugin_category          str
Duration_category        str
dtype: object
['session_ID', 'Garage_ID', 'User_ID', 'User_type', 'Shared_ID', 'Start_plugin', 'Start_plugin_hour', 'End_plugout', 'End_plugout_hour', 'El_kWh', 'Duration_hours', 'month_plugin', 'weekdays_plugin', 'Plugin_category', 'Duration_category']


In [38]:
traffic_reports = pd.read_csv('Dataset 6_Local traffic distribution.csv', sep=None, engine='python')
traffic_reports['Date_from'] = pd.to_datetime(traffic_reports['Date_from'], dayfirst=True).dt.hour

ev_charging_traffic = data.merge(traffic_reports, left_on='Start_plugin_hour', right_on='Date_from', how='left')
ev_charging_traffic.head()


,session_ID,Garage_ID,User_ID,User_type,Shared_ID,Start_plugin,Start_plugin_hour,End_plugout,End_plugout_hour,El_kWh,...,weekdays_plugin,Plugin_category,Duration_category,Date_from,Date_to,KROPPAN BRU,MOHOLTLIA,SELSBAKK,MOHOLT RAMPE 2,Jonsvannsveien vest for Steinanvegen
0,1,AdO3,AdO3-4,Private,NaN,21.12.2018 10:20,10,21.12.2018 10:23,10.0,"0,3",...,Friday,late morning (9-12),Less than 3 hours,10,01.12.2018 11:00,1773,1015,327,112,425
1,1,AdO3,AdO3-4,Private,NaN,21.12.2018 10:20,10,21.12.2018 10:23,10.0,"0,3",...,Friday,late morning (9-12),Less than 3 hours,10,02.12.2018 11:00,1111,534,204,55,307
2,1,AdO3,AdO3-4,Private,NaN,21.12.2018 10:20,10,21.12.2018 10:23,10.0,"0,3",...,Friday,late morning (9-12),Less than 3 hours,10,03.12.2018 11:00,2457,1177,363,119,511
3,1,AdO3,AdO3-4,Private,NaN,21.12.2018 10:20,10,21.12.2018 10:23,10.0,"0,3",...,Friday,late morning (9-12),Less than 3 hours,10,04.12.2018 11:00,2397,1100,345,120,483
4,1,AdO3,AdO3-4,Private,NaN,21.12.2018 10:20,10,21.12.2018 10:23,10.0,"0,3",...,Friday,late morning (9-12),Less than 3 hours,10,05.12.2018 11:00,2342,1222,351,97,457


In [39]:
print(ev_charging_traffic.shape)
print()
print(ev_charging_traffic.isnull().sum())
print()
print(ev_charging_traffic.describe())


(2936906, 22)

session_ID                                    0
Garage_ID                                     0
User_ID                                       0
User_type                                     0
Shared_ID                               2333982
Start_plugin                                  0
Start_plugin_hour                             0
End_plugout                               14518
End_plugout_hour                          14518
El_kWh                                        0
Duration_hours                            14518
month_plugin                                  0
weekdays_plugin                               0
Plugin_category                               0
Duration_category                         14518
Date_from                                     0
Date_to                                       0
KROPPAN BRU                                   0
MOHOLTLIA                                     0
SELSBAKK                                      0
MOHOLT RAMPE 2           

In [40]:
cols_to_drop = ['session_ID', 'Garage_ID', 'User_ID', 'Shared_ID', 'Plugin_category',
                'Duration_category', 'Start_plugin', 'Start_plugin_hour', 'End_plugout',
                'End_plugout_hour', 'Date_from', 'Date_to',
                'User_type', 'month_plugin', 'weekdays_plugin']

ev_charging_traffic.drop(columns=cols_to_drop, inplace=True)
ev_charging_traffic.head()


,El_kWh,Duration_hours,KROPPAN BRU,MOHOLTLIA,SELSBAKK,MOHOLT RAMPE 2,Jonsvannsveien vest for Steinanvegen
0,"0,3","0,05",1773,1015,327,112,425
1,"0,3","0,05",1111,534,204,55,307
2,"0,3","0,05",2457,1177,363,119,511
3,"0,3","0,05",2397,1100,345,120,483
4,"0,3","0,05",2342,1222,351,97,457


In [41]:
ev_charging_traffic[['El_kWh', 'Duration_hours']] = ev_charging_traffic[['El_kWh', 'Duration_hours']].apply(lambda col: col.str.replace(',', '.'))

ev_charging_traffic = ev_charging_traffic.apply(pd.to_numeric, errors='coerce').astype(float)
ev_charging_traffic.dtypes


El_kWh                                  float64
Duration_hours                          float64
KROPPAN BRU                             float64
MOHOLTLIA                               float64
SELSBAKK                                float64
MOHOLT RAMPE 2                          float64
Jonsvannsveien vest for Steinanvegen    float64
dtype: object

In [42]:
from sklearn.model_selection import train_test_split

ev_charging_traffic.dropna(inplace=True)

X = ev_charging_traffic.drop(columns=['El_kWh'])
y = ev_charging_traffic['El_kWh']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)


(2337283, 6) (584321, 6)
(2337283,) (584321,)


In [43]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [44]:
from sklearn.metrics import mean_squared_error

test_mse = mean_squared_error(y_test, lr_model.predict(X_test))
print(test_mse)


135.47014475793122


In [45]:
import torch
from torch import nn, optim


In [46]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float, device=device)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float, device=device)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float, device=device)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float, device=device)

torch.manual_seed(42)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(42)

model = nn.Sequential(
    nn.Linear(X_train.shape[1], 56),
    nn.ReLU(),
    nn.Linear(56, 26),
    nn.ReLU(),
    nn.Linear(26, 1)
).to(device)

print(model)


Using device: cuda
Sequential(
  (0): Linear(in_features=6, out_features=56, bias=True)
  (1): ReLU()
  (2): Linear(in_features=56, out_features=26, bias=True)
  (3): ReLU()
  (4): Linear(in_features=26, out_features=1, bias=True)
)


In [47]:
loss = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0007)


In [48]:
epochs = 3000

for epoch in range(1, epochs + 1):
    model.train()

    y_pred = model(X_train_tensor).squeeze(1)
    train_mse = loss(y_pred, y_train_tensor)

    optimizer.zero_grad()
    train_mse.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(f'Epoch {epoch}/{epochs} - MSE: {train_mse.item():.6f}')

model_path = 'ev_charging_nn_3000_epochs.pth'
torch.save(model.state_dict(), model_path)
print(f'Model saved to {model_path}')


Epoch 500/3000 - MSE: 148.593948
Epoch 1000/3000 - MSE: 138.640717
Epoch 1500/3000 - MSE: 131.162613
Epoch 2000/3000 - MSE: 125.889732
Epoch 2500/3000 - MSE: 122.734467
Epoch 3000/3000 - MSE: 121.125168
Model saved to ev_charging_nn_3000_epochs.pth


In [49]:
model.eval()
with torch.no_grad():
    y_test_pred = model(X_test_tensor).squeeze(1)
    test_loss = loss(y_test_pred, y_test_tensor)

print(test_loss.item())


120.73092651367188


In [50]:
nn_mse_3000 = test_loss.item()

torch.manual_seed(42)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(42)

model_5000 = nn.Sequential(
    nn.Linear(X_train.shape[1], 56),
    nn.ReLU(),
    nn.Linear(56, 26),
    nn.ReLU(),
    nn.Linear(26, 1)
).to(device)

loss_5000 = nn.MSELoss()
optimizer_5000 = optim.Adam(model_5000.parameters(), lr=0.0007)

epochs_5000 = 5000
for epoch in range(1, epochs_5000 + 1):
    model_5000.train()
    y_pred_5000 = model_5000(X_train_tensor).squeeze(1)
    train_mse_5000 = loss_5000(y_pred_5000, y_train_tensor)

    optimizer_5000.zero_grad()
    train_mse_5000.backward()
    optimizer_5000.step()

    if epoch % 500 == 0:
        print(f'Epoch {epoch}/{epochs_5000} - MSE: {train_mse_5000.item():.6f}')

model_5000.eval()
with torch.no_grad():
    y_test_pred_5000 = model_5000(X_test_tensor).squeeze(1)
    test_loss_5000 = loss_5000(y_test_pred_5000, y_test_tensor)

print('\nComparison:')
print(f'Linear Regression test MSE: {test_mse:.6f}')
print(f'Neural Net (3000 epochs) test MSE: {nn_mse_3000:.6f}')
print(f'Neural Net (5000 epochs) test MSE: {test_loss_5000.item():.6f}')

model_path_5000 = 'ev_charging_nn_5000_epochs.pth'
torch.save(model_5000.state_dict(), model_path_5000)
print(f'Model saved to {model_path_5000}')


Epoch 500/5000 - MSE: 148.593948
Epoch 1000/5000 - MSE: 138.640717
Epoch 1500/5000 - MSE: 131.162613
Epoch 2000/5000 - MSE: 125.889732
Epoch 2500/5000 - MSE: 122.734467
Epoch 3000/5000 - MSE: 121.125168
Epoch 3500/5000 - MSE: 120.469460
Epoch 4000/5000 - MSE: 132.212769
Epoch 4500/5000 - MSE: 119.653160
Epoch 5000/5000 - MSE: 125.268539

Comparison:
Linear Regression test MSE: 135.470145
Neural Net (3000 epochs) test MSE: 120.730927
Neural Net (5000 epochs) test MSE: 135.059631
Model saved to ev_charging_nn_5000_epochs.pth
